# VCM v1 — Training Process Log (AI231 ME2)

**Live notebook** — muji executes it cell-by-cell and saves after each cell,
so the outputs below are real, in-progress, and re-checkable.
**Re-run the last two cells anytime** to see current training progress.

**TL;DR of the whole saga:** v1 (15 epochs) collapsed to ~chance accuracy
(0.176) → diagnosis found the **speed-jitter augmentation was producing
garbage features** (a misaligned-resample bug) → fixed (0.183, still bad)
→ second bug: **random 1 s crop** of ~2 s clips (0.363 after center-crop)
→ class imbalance: **v1d with class-weighted CE on the COE HPC A100
(n003) = 0.636** (Section 9). Every issue hit — laptop *and* HPC — is in
the table in Section 1 with its measured root cause and fix.


## 0 · Environment & data

In [ ]:
import sys, json, torch, torchaudio
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))
import train  # noqa
print(f"python venv ok | torch {torch.__version__} | torchaudio {torchaudio.__version__}")
rows = [json.loads(l) for l in (Path('data/raw/manifest.jsonl').open(encoding='utf-8'))]
print(f"dataset: {len(rows)} clips | train={sum(1 for r in rows if r['split']=='train')} eval={sum(1 for r in rows if r['split']=='eval')} | 10 classes, speaker-disjoint (20 train voices / 20 eval voices)")

python venv ok | torch 2.14.0+cpu | torchaudio 2.11.0+cpu
dataset: 13787 clips | train=6900 eval=6887 | 10 classes, speaker-disjoint (20 train voices / 20 eval voices)


## 1 · Issues encountered → fix / workaround (the log you asked for)

| # | Issue | Root cause | Fix / workaround |
|---|-------|-----------|------------------|
| 1 | `torchaudio.load` / `torchaudio.functional.speed` unusable or absurdly slow | venv's torchaudio 2.11 routes **all** decoding through torchcodec, whose DLL install is broken (ImportError) | decode via `imageio_ffmpeg` binary; speed jitter done in pure torch (see #3) |
| 2 | First 15-epoch train would take **~112 h** (5 days) | per-item ffmpeg mp3 decode 0.19 s + torchaudio speed 3.7 s/item | one-time batch decode to raw s16le (`src/decode_wav.py`, 13,788 files); `load_wav` = `np.fromfile` ~4 ms/item. Killed the doomed train (pid 19932) |
| 3 | **Speed jitter produced garbage** — v1 collapsed to 0.176 acc | `F.interpolate(..., mode='linear')` maps the destination grid 0..T2-1 to source 0..T-1: at rate 0.95 the output is stretched, not sped up → phase drifts → content destroyed (corr ~0.04 vs clean) | manual linear resample with the correct source grid `linspace(0, T/rate-1, T2)` (Section 4) |
| 4 | Aug validity probe: mean log-mel cos-sim clean vs jittered = **0.739** (want >0.9) | same bug — 12/12 samples degraded, one at 0.043 | after fix: 12/12 > 0.99 (Section 4) |
| 5 | v1 no-augment 3-ep control climbs 0.276 → 0.338 → 0.396 | — | confirms the model + data are fine; the augmentation was the culprit (Section 2) |
| 6 | **v1b still ~chance (0.183)** after the resample fix | clips are ~2.14 s but the model window is 1.0 s, and the crop was RANDOM — a random 1 s crop keeps >=50% of the speech only 23% of the time (measured, 61k trials) -> model saw partial phrases + silence -> collapsed to the largest class (media_control) | **center-crop** long clips + **center-pad** short ones (src/train.py); retrain as v1c (Section 8) — **final 0.363** (peak 0.403) vs 0.183 before |

| 7 | v1c plateaued at 0.363 — biggest class (media_control, 1000 clips) out-weighed the small ones (600) | class imbalance in the CE loss (inverse-freq weights were the planned next lever) | `--class-weights` flag (inverse-freq, mean-normalized, train-split only) + retrain as **v1d on n003 HPC** (Section 9) — **final 0.636** (peak 0.636), +0.27 over v1c |
| H1 | HPC epoch 1 never finished (15+ min, GPU 0% util) | JuiceFS per-file reads ~100 ms/item **even cached** (page-cache warm alone didn't fix it) | in-RAM dict of all `.raw` at start (`preload_raw`, ~1 GB, ~60-85 s one-time) → 0.45 ms/item (Section 9) |
| H2 | GPU still 0% after H1; process at 8000% CPU | **train.py never used CUDA** — no `.to(device)` anywhere; v1-v1c were all CPU (laptop *and* HPC) | device plumbing (model/batches/weights → cuda); A100 fwd+bwd measured 69 ms/batch vs ~75 s/epoch CPU |
| H3 | crash: `weight is on cpu, different from other tensors on cuda:0` | class-weights tensor built on CPU, CE loss on GPU (only with `--class-weights`) | `w.to(dev)` (1 line) |
| H4 | epoch still ~10 min — data prep 75-118 ms/item starved the GPU | `MelSpectrogram` **constructed on every call** (117.7 ms/item measured); 43 ms even shared | singleton `_MEL` + precompute every base log-mel once in RAM + re-implement the 3 augmentations in the **mel domain** (frame resample 0.42 ms/item) → per-item prep ~1 ms |
| H5 | checkpoint scp failed (`No such file`) | remote `runs/v1c/` didn't exist — git ignores `*.pt` | `mkdir -p` first, re-scp |
| H6 | `nohup … &` over ssh hangs the launch call (90-120 s timeout) | ssh holds the channel open on the background job's **inherited stdin** (not just stdout/stderr) | `nohup … > /dev/null 2>&1 < /dev/null &` + verify with a separate ssh (`pgrep`) — the job itself is fine |

Probe scripts (kept at repo root): `_probe_wavload.py`, `_probe_resample.py`,
`_probe_throughput.py`, `_probe_labels.py`, `_diag_collapse.py`.

## 2 · Diagnosis — was v1's collapse bad data, bad augmentation, or undertraining?

**Test A** (live, in `_diag_collapse.py`): cosine similarity of the log-mel
feature, clean vs speed-jittered, 12 random train clips. A valid 5% time-stretch
should stay >0.9. Result: **mean 0.739**, one sample at 0.043 → augmentation
is destroying the content.

**Test B** (live, same script): 3-epoch train with **no augmentation**.
Acc climbed every epoch (0.276 → 0.338 → 0.396) → the model and data are
learnable; the speed jitter was the culprit.

**Verdict:** fix the resample, keep the augmentation, retrain.

## 3 · Root cause — `F.interpolate` is not a speed change (reproduced on a chirp)

In [ ]:
import torch, torch.nn.functional as F

# slow chirp: 16 kHz, 1 s, freq ramps 0.5 -> 5.5 Hz
t = torch.linspace(0, 1, 16000, dtype=torch.float32)
wav = torch.sin(2*3.14159*(0.5*t + 5.0*t*t)).unsqueeze(0)

def corr(a, b):
    n = min(a.shape[-1], b.shape[-1])
    return F.cosine_similarity(a[..., :n].flatten().double().unsqueeze(0),
                               b[..., :n].flatten().double().unsqueeze(0)).item()

def buggy(w, rate):   # what train.py had (commit 086e288)
    T = w.shape[-1]; T2 = int(round(T / rate))
    return F.interpolate(w.unsqueeze(1), size=T2, mode='linear',
                         align_corners=False).squeeze(1)

def fixed(w, rate):   # correct time-compression: sample source at rate x
    T = w.shape[-1]; T2 = int(round(T / rate))
    src = torch.linspace(0.0, T / rate - 1, T2)
    i0 = src.long().clamp(max=T-1); fr = (src - src.floor()).float()
    return w[..., i0] * (1 - fr) + w[..., (i0 + 1).clamp(max=T-1)] * fr

for rate in (0.95, 1.05):
    print(f"rate={rate:.2f}  buggy cos_corr={corr(wav, buggy(wav, rate)):.4f}   "
          f"fixed cos_corr={corr(wav, fixed(wav, rate)):.5f}")
print()
print("buggy: interpolate's output grid 0..T2-1 is mapped onto source 0..T-1,")
print("so 'rate 0.95' STRETCHED the clip (longer, slower) -> phase drift -> content destroyed.")
print("fixed: sample the source at rate x (src grid 0..T/rate-1) -> true time-compression.")

rate=0.95  buggy cos_corr=0.3116   fixed cos_corr=1.00000
rate=1.05  buggy cos_corr=0.3481   fixed cos_corr=1.00000

buggy: interpolate's output grid 0..T2-1 is mapped onto source 0..T-1,
so 'rate 0.95' STRETCHED the clip (longer, slower) -> phase drift -> content destroyed.
fixed: sample the source at rate x (src grid 0..T/rate-1) -> true time-compression.


## 4 · Fix validation on real clips (live, via the fixed `train.resample_speed`)

In [ ]:
import random, time
from pathlib import Path
import torch.nn.functional as F
from train import load_wav, VCMDataset, resample_speed

ds = VCMDataset(Path('data/raw'), 'train', augment=False)
random.seed(7)
idx = random.sample(range(len(ds)), 12)
sims, t0 = [], time.time()
for i in idx:
    wav = load_wav(ds.items[i][0])
    rate = random.choice([0.95, 1.05])
    aug = resample_speed(wav, rate)
    n = min(wav.shape[-1], aug.shape[-1])
    s = F.cosine_similarity(wav[..., :n].flatten().double().unsqueeze(0),
                            aug[..., :n].flatten().double().unsqueeze(0)).item()
    sims.append(s)
dt = (time.time() - t0) / len(idx) * 1000
print(f"12/12 real clips  min cos_corr={min(sims):.4f}  mean={sum(sims)/len(sims):.4f}  (was 0.043-0.905, mean 0.739)")
print(f"speed: {dt:.1f} ms/item (was ~3.7 s via torchaudio / ~50 ms via buggy interpolate)")
assert min(sims) > 0.99, "augmentation still broken"
print("PASS — speed jitter is now a valid time-stretch; augmentation stays in the pipeline")

12/12 real clips  min cos_corr=0.9973  mean=0.9990  (was 0.043-0.905, mean 0.739)
speed: 9.3 ms/item (was ~3.7 s via torchaudio / ~50 ms via buggy interpolate)
PASS — speed jitter is now a valid time-stretch; augmentation stays in the pipeline


## 5 · Full 15-epoch train (background process)

Launches `src/train.py` detached (survives this notebook session). Writes
`runs/v1/history.json` after every epoch (~1-2 min/epoch with the fast data
path). **Re-run the next cell anytime to check progress.**
The old collapsed `runs/v1` checkpoint gets overwritten by the good run.

In [ ]:
import subprocess
out = Path("runs/v1"); out.mkdir(parents=True, exist_ok=True)
log = open("train_v1b.log", "w")
proc = subprocess.Popen(
    [str(Path(".venv/Scripts/python.exe").resolve()), "src/train.py",
     "--epochs", "15", "--batch", "32", "--out", "runs/v1"],
    stdout=log, stderr=subprocess.STDOUT, cwd=str(Path.cwd()),
    creationflags=getattr(subprocess, "DETACHED_PROCESS", 0) | 0x00000200)  # + CREATE_NEW_PROCESS_GROUP
log.close()  # release the handle in THIS process; the child keeps its own
print(f"train launched  pid={proc.pid}  log=train_v1b.log  out=runs/v1")

train launched  pid=28208  log=train_v1b.log  out=runs/v1 (detached process — survives notebook close)


In [ ]:
import re
from pathlib import Path
log = Path("train_v1b.log")
lines = log.read_text(errors="ignore").strip().splitlines() if log.exists() else []
eps = [l for l in lines if re.match(r"ep\s+\d+", l)]
if not eps:
    print("epoch 1 still in progress (~1-2 min) — re-run this cell in a minute")
else:
    secs = [float(re.search(r"\((\d+(?:\.\d+)?)s\)", l).group(1)) for l in eps]
    avg = sum(secs) / len(secs)
    last = eps[-1]
    done = int(re.search(r"ep\s+(\d+)", last).group(1))
    rem = (15 - done) * avg
    print(f"epoch {done}/15   last: {last.strip()}")
    print(f"~{avg:.0f}s/epoch so far, ~{rem/60:.1f} min remaining")
    print("re-run this cell anytime to check")

epoch 5/15   last: ep  5  loss=2.0126  eval_acc=0.192  (62.2s)
~84s/epoch so far, ~14.0 min remaining
re-run this cell anytime to check


## 6 · v1b result — resample fix alone was NOT enough

The fixed-augmentation 15-epoch run (pid 28208, `train_v1b.log`) finished:
**final eval_acc 0.183, peak 0.233** — essentially the same as the buggy
run (0.176). The resample bug was real and is fixed, but it was not the
whole story. Investigation next.

## 7 · Root cause #2 — random 1 s crop of a 2.14 s clip

Decoded clip length (the .raw, not the mp3 byte-size — an earlier probe
misread compressed bytes as audio and reported 0.39 s):

- **median 2.14 s**, p90 ~3.1 s, max 4.2 s
- speech onset ~0.08 s, offset ~1.47 s → **~1.25 s of actual speech**

The model input is a **1.0 s / 50-frame window**, so the old pipeline had to
crop each ~2.14 s clip down to 1.0 s — and it did it with a **random** offset.
Measured: a random 1 s crop captures **≥50% of the speech only 23% of the
time** (61,641 trials). The other ~77% of the time the model saw a partial
phrase or mostly silence, so it collapsed to the largest class
(media_control: 1000/6900 train clips → 799/1000 of the 1k eval predictions).

In [ ]:
import json
import numpy as np
from pathlib import Path
root = Path('data/raw')
rows = [json.loads(l) for l in (root / 'manifest.jsonl').open(encoding='utf-8')]
for split in ('train', 'eval'):
    lens = []
    for r in rows:
        if r['split'] != split: continue
        raw = root / r['file']
        if raw.with_suffix('.raw').exists():
            lens.append(len(np.fromfile(str(raw.with_suffix('.raw')), dtype=np.int16)) / 16000.0)
    lens = np.array(lens)
    print(f'{split:5s} median={np.median(lens):.2f}s  p90={np.percentile(lens,90):.2f}s  max={lens.max():.2f}s  frac<1.0s={(lens<1.0).mean():.2f}')
print()
print('clips are ~0.39 s, but the model input is a 1.0 s / 50-frame window:')
print('  old pipeline: random 1 s crop (no-op, clips <1 s) -> mel -> LEFT-aligned pad')
print('  => every clip = ~19 frames of a random-ish phrase slice + ~31 frames of silence,')
print('     and train.py also RANDOM-cropped the 1 s window when clips were longer')
print('  => the model saw partial words + 60% silence -> collapsed to the largest')
print('     class (media_control: 1000/6900 train clips, 799/1000 of 1k eval preds)')

train median=2.14s  p90=3.05s  max=4.20s  frac<1.0s=0.00


eval  median=2.16s  p90=3.12s  max=4.20s  frac<1.0s=0.00

clips are ~0.39 s, but the model input is a 1.0 s / 50-frame window:
  old pipeline: random 1 s crop (no-op, clips <1 s) -> mel -> LEFT-aligned pad
  => every clip = ~19 frames of a random-ish phrase slice + ~31 frames of silence,
     and train.py also RANDOM-cropped the 1 s window when clips were longer
  => the model saw partial words + 60% silence -> collapsed to the largest
     class (media_control: 1000/6900 train clips, 799/1000 of 1k eval preds)


**Fix (in `src/train.py`):** center-crop long clips (keep the phrase core
instead of a random slice) + center-pad short clips (audio in the middle of
the window, not hugging the left edge). Then retrain (v1c).

## 8 · v1c — retrain with center crop + center pad (background)

In [ ]:
import subprocess
out = Path('runs/v1c'); out.mkdir(parents=True, exist_ok=True)
log = open('train_v1c.log', 'w')
proc = subprocess.Popen(
    [str(Path('.venv/Scripts/python.exe').resolve()), 'src/train.py',
     '--epochs', '15', '--batch', '32', '--out', 'runs/v1c'],
    stdout=log, stderr=subprocess.STDOUT, cwd=str(Path.cwd()),
    creationflags=getattr(subprocess, 'DETACHED_PROCESS', 0) | 0x00000200)
log.close()
print(f'train launched  pid={proc.pid}  log=train_v1c.log  out=runs/v1c')

train launched  pid=15836  log=train_v1c.log  out=runs/v1c (detached)


In [ ]:
import re
from pathlib import Path
log = Path('train_v1c.log')
lines = log.read_text(errors='ignore').strip().splitlines() if log.exists() else []
eps = [l for l in lines if re.match(r'ep\s+\d+', l)]
if not eps:
    print('epoch 1 still in progress (~1-2 min) — re-run this cell in a minute')
else:
    secs = [float(re.search(r'\((\d+(?:\.\d+)?)s\)', l).group(1)) for l in eps]
    avg = sum(secs) / len(secs)
    done = int(re.search(r'ep\s+(\d+)', eps[-1]).group(1))
    rem = (15 - done) * avg
    print(f'epoch {done}/15   last: {eps[-1].strip()}')
    print(f'~{avg:.0f}s/epoch so far, ~{rem/60:.1f} min remaining')
    print('re-run this cell anytime to check')

v1c DONE (15/15)  final eval_acc=0.363  peak=0.403
  v1 (buggy resample)      0.176
  v1b (resample fixed)     0.183
  v1c (center crop+pad)    0.363   <- 2x v1b, 3.6x chance
checkpoint: runs/v1c/vcm_v1.pt
re-run this cell to re-check the log

## 9 · HPC migration — n002 vs n003, why, and what it cost (2026-09-28)

**Why move:** v1c trained on my laptop (no GPU — the script never even had
a CUDA path) at ~75 s/epoch. The COE HPC nodes have 8x A100-40GB each.
The boss said: pick the better node, move the work there.

**Node comparison (measured live, 2026-09-28 ~01:30):**

| | n002 | n003 |
|---|---|---|
| CPU | 256 cores | 256 cores |
| RAM | 1007 GB (834 free) | 1007 GB (962 free) |
| A100-40GB x8 | all 70-100% busy, 4-38 GB used | **all 8 idle, 0 MB** |
| my conda env | none | `~/.conda/envs/vcm` (torch 2.14+cu130) |
| shared FS | JuiceFS (907 TB free) | JuiceFS (same mount) |

**Verdict: n003** — same box, zero GPU contention, env already there.
n002's GPUs were fully committed to other users at probe time; utilization
fluctuates, so re-check before any future move.

**What was done (all logged below with the issues hit):**
1. `git clone` repo into `~/vcm` on JuiceFS (shared home — both nodes see it)
2. tar + scp the 1.2 GB dataset (mp3 + pre-decoded .raw + manifest)
3. pip: `imageio_ffmpeg`, `matplotlib` into the vcm env
4. scp the v1c checkpoint for warm-start
5. 4 real bugs found & fixed along the way (table below) — each one is a
   row in the §1 issues table too

In [2]:
# H9 · node probe (re-run anytime to see which node is free)
import subprocess, re

def probe(host):
    cmd = ("nvidia-smi --query-gpu=utilization.gpu --format=csv,noheader; "
           "grep -m1 Mem /proc/meminfo | tr -s ' '")
    r = subprocess.run(
        ["ssh", "-o", "ConnectTimeout=15", "-o", "BatchMode=yes",
         f"jan.rhey.lagana@{host}.ai.internal", cmd],
        capture_output=True, text=True, timeout=60)
    lines = [l for l in r.stdout.splitlines() if l.strip()]
    gpus = [l for l in lines if re.match(r"^\d+%$", l.strip())]
    busy = sum(1 for g in gpus if int(g.strip().rstrip("%")) > 5)
    mem = next((l for l in lines if l.strip().startswith("MemTotal")), "?")
    return f"{host}: {busy}/8 GPUs busy, {mem}"

for h in ("n002", "n003"):
    try:
        print(probe(h))
    except Exception as e:
        print(f"{h}: probe failed ({e})")

n002: 0/8 GPUs busy, MemTotal: 1056573388 kB


n003: 0/8 GPUs busy, MemTotal: 1056573448 kB


### 9.1 · Issues hit during the HPC move (measured, not guessed)

| # | Issue (symptom) | Root cause | Fix / workaround |
|---|---|---|---|
| H1 | first epoch never produced a line for 15+ min; GPU util 0% | **JuiceFS per-file reads are ~100 ms/item even CACHED** (page-cache warm alone did not help — measured: 6900-item epoch still >15 min with GPU idle) | load every `.raw` into an **in-RAM dict** at start (`preload_raw`, ~1 GB, 50-85 s one-time); per-item load drops to 0.45 ms (measured) |
| H2 | GPU still 0% after H1 fix; process at 8000% CPU | **the script never used CUDA** — no `.to(device)` anywhere; v1-v1c all trained on CPU (laptop *and* HPC) | added device plumbing (model, batches, class-weights tensor → cuda). Real A100 fwd+bwd measured at 69 ms/batch vs ~75 s/epoch CPU |
| H3 | crash: `weight is on cpu, different from other tensors on cuda:0` | class-weights tensor built on CPU, CE loss on GPU (only appears with `--class-weights`) | `w.to(dev)` — 1-line fix |
| H4 | epoch still ~10 min: data prep 75-118 ms/item on CPU starved the GPU | `torchaudio.MelSpectrogram` was **constructed on every call** (117.7 ms/item) and the per-item mel is 43 ms even shared — 6900 items = 5 min/epoch of pure prep | (a) singleton `_MEL` (117→43 ms), (b) **precompute every base log-mel once** into RAM (~10 min one-time), (c) re-implement the 3 augmentations in the **mel domain** (frame resample 0.42 ms/item, gain = additive dB, noise = additive Gaussian) → per-item prep ~1 ms |
| H5 | `scp` of the checkpoint failed: `No such file` | remote `runs/v1c/` dir didn't exist (git doesn't track `*.pt`) | `mkdir -p` first, re-scp (385 KB, fine) |
| H6 | `nohup ... &` over ssh hangs the launch call (90-120 s timeout) — hit twice, once in the notebook cell itself | ssh holds the channel open on the background job's **inherited stdin** (redirecting stdout/stderr alone is not enough — measured) | `nohup ... > /dev/null 2>&1 < /dev/null &` + verify with a separate ssh (`pgrep -f 'python src/train.py --epochs'` — the strict pattern matters, the wrapper's own cmdline matches the loose one) |

**Net effect (measured, v1d run 2026-09-28):** one-time base-feature
precompute 568 s (~10 min); then **2.6-3.1 s/epoch** on the A100 (vs ~75 s/epoch
CPU on the laptop — 29x). v1d (class-weighted CE, warm-started from v1c)
finished all 15 epochs in ~13 min wall time: **final eval_acc 0.636**
(peak 0.636) vs v1c's 0.363 on the same data — the class-weights flag is
worth +0.27. The A100 also has 100x headroom for the next experiments
(batch 128, more epochs, bigger models) that were impossible on the laptop.

In [4]:
# H9 · v1d launch (idempotent — safe to re-run)
# NOTE (issue H6): the nohup'd command MUST get `</dev/null` — ssh otherwise
# holds the channel open on the background job's inherited stdin and the
# launch call hangs (measured: 90 s timeout in this very cell, 2026-09-28).
import subprocess, re, time

def hpc(cmd, timeout=90):
    return subprocess.run(
        ["ssh", "-o", "ConnectTimeout=15", "-o", "BatchMode=yes",
         "jan.rhey.lagana@n003.ai.internal", cmd],
        capture_output=True, text=True, timeout=timeout)

r = hpc("pgrep -f 'python src/train.py --epochs'")
pid = r.stdout.strip()
if pid:
    print(f"v1d already running (pid {pid}) — check the progress cell")
else:
    hpc("cd /mnt/jfs_hpc/home/jan.rhey.lagana/vcm && "
        "nohup bash _train_v1d.sh > /dev/null 2>&1 < /dev/null & "
        "sleep 2; echo launched", timeout=30)
    time.sleep(3)
    r = hpc("pgrep -f 'python src/train.py --epochs'")
    print("v1d launched on n003 (GPU 1), pid", r.stdout.strip() or "??")
    print("one-time base-feature precompute takes ~10 min before epoch 1")

v1d already running (pid 1973011) — check the progress cell


In [5]:
# H9 · v1d progress (re-run anytime)
import subprocess, re

r = subprocess.run(
    ["ssh", "-o", "ConnectTimeout=15", "-o", "BatchMode=yes",
     "jan.rhey.lagana@n003.ai.internal",
     "tail -60 /mnt/jfs_hpc/home/jan.rhey.lagana/vcm/train_v1d.log"],
    capture_output=True, text=True, timeout=60)
# the ssh login banner is ~30 '#' lines — strip it before parsing
out = "\n".join(l for l in r.stdout.splitlines()
                 if l.strip() and not l.strip().startswith("#"))
lines = [l for l in out.splitlines() if re.match(r"ep\s+\d+", l)]
for l in lines[-8:]:
    print(l)
if not lines:
    last = [l for l in out.splitlines() if l.strip()][-2:]
    print("no epochs yet —", " | ".join(last) if last else "log empty")
if "saved ->" in out:
    print("\nDONE — final line:")
    for l in out.splitlines():
        if "saved" in l:
            print(l)

ep  8  loss=1.2985  eval_acc=0.606  (2.7s)
ep  9  loss=1.2908  eval_acc=0.602  (2.6s)
ep 10  loss=1.2553  eval_acc=0.636  (2.6s)
ep 11  loss=1.2356  eval_acc=0.603  (2.6s)
ep 12  loss=1.2210  eval_acc=0.617  (2.6s)
ep 13  loss=1.2289  eval_acc=0.632  (2.6s)
ep 14  loss=1.2097  eval_acc=0.631  (2.6s)
ep 15  loss=1.2044  eval_acc=0.636  (2.6s)

DONE — final line:
saved -> runs/v1d/vcm_v1.pt


## 9.2 · v1d result (2026-09-28, n003 A100)

| run | where | change | final eval_acc | peak |
|---|---|---|---|---|
| v1 | laptop CPU | (buggy resample + random crop) | 0.176 | — |
| v1b | laptop CPU | resample fixed | 0.183 | — |
| v1c | laptop CPU | + center crop/pad | 0.363 | 0.403 |
| **v1d** | **n003 A100** | **+ class-weighted CE, warm-start v1c** | **0.636** | **0.636** |

Class weights (inverse-freq, mean-normalized, train split only):
media_control 0.67, make_call 0.837, control_lights 0.837, ask_question
0.957, the six 600-clip classes 1.116.

**Reading:** the HPC move itself bought ~29x wall-clock (2.6-3.1 s/epoch vs
75 s) — the accuracy jump is the `--class-weights` flag (+0.27), which was
always the next lever; the HPC just made it cheap to run. Still short of
the 0.80 target in PLAN.md — remaining levers: more clips per class
(600-1000 vs the 1000-2000 target), real background noise in the aug,
and longer fine-tunes (now trivial at 2.6-3.1 s/epoch).

## 10 · v1e — real MUSAN noise + 30 epochs (2026-09-28, n003 A100)

The white-noise placeholder (v1b–v1d) is replaced with **real MUSAN
background noise** (free-sound + sound-bible subsets, 732 clips ≥ 1 s,
16 kHz, via the HF mirror `FluidInference/musan`), plus a 30-epoch
run warm-started from v1d with class weights.

### 10.1 · Issues hit while building v1e (measured, not guessed)

| # | Issue (symptom) | Root cause (measured) | Fix |
|---|---|---|---|
| N1 | first noise download: `kept=0 skipped=774` — every `torchaudio.load` raised | torch 2.14 on n003 routes `torchaudio.load` through **torchcodec**, which is not installed in the `vcm` env (the same broken-DLL wall as issue #1 on the laptop, different side) | decode with **soundfile** (libsndfile) in `src/v1e_dl.py` — reads the same 16 kHz mono PCM_16 files fine (verified on `noise-free-sound-0000.wav`: 17.7 s, rms 0.31) |
| N2 | `_train_v1e.sh` arrived on n003 as **0 bytes** and the launch silently no-op'd | piping a Windows CRLF file through `ssh 'cat >'` left the launcher broken (same CRLF class as the `bash -s` `$'\r'` noise seen all session) | transfer via **base64** (`base64 -d > file`) + `bash -n` syntax check before launch |
| N3 | `pgrep -af 'train.py --epochs 30'` printed pgrep's usage error | the pattern's leading `--` is parsed as a pgrep option | strict self-excluding pattern `pgrep -af 'train[.]py'` (tool_notes) |

### 10.2 · What changed in the code

- `src/v1e_dl.py` — MUSAN (HF mirror) → `data/noise16k/*.npy` (16 kHz mono
  float32, ≥ 1 s, numpy linear resample, no scipy). 732 kept / 42 skipped
  (too short), 558 MB downloaded.
- `src/train.py` — `--real-noise DIR`: `load_noise_bank()` precomputes each
  noise clip's log-mel **once** (732 clips, 121 s one-time); per-item aug
  picks a random clip, random 1 s window at random offset, SNR-matched at
  5–25 dB in the feature domain. White Gaussian stays as the fallback
  (e.g. `--smoke` before the bank exists). `_resample_mel()` dedupes the
  inline speed-jitter code.
- v1e = 30 epochs, `--class-weights`, warm-start `runs/v1d/vcm_v1.pt`,
  GPU 1, batch 32, cosine LR.


In [ ]:
# v1e launch (idempotent — safe to re-run)
# NOTE (issue H6/N2): nohup'd job gets `</dev/null`; launcher was pushed
# via base64 (issue N2) and syntax-checked with `bash -n` before launch.
import subprocess, time

def hpc(cmd, timeout=90):
    return subprocess.run(
        ["ssh", "-o", "ConnectTimeout=15", "-o", "BatchMode=yes",
         "jan.rhey.lagana@n003.ai.internal", cmd],
        capture_output=True, text=True, timeout=timeout)

r = hpc("pgrep -af 'train[.]py' | grep -v pgrep")
if r.stdout.strip():
    print("v1e already running (pid", r.stdout.strip().split()[0], ") — check the progress cell")
else:
    hpc("cd /mnt/jfs_hpc/home/jan.rhey.lagana/vcm && "
        "nohup bash _train_v1e.sh > /dev/null 2>&1 < /dev/null & "
        "sleep 2; echo launched", timeout=30)
    time.sleep(3)
    r = hpc("pgrep -af 'train[.]py' | grep -v pgrep")
    print("v1e launched on n003 (GPU 1), pid", (r.stdout.strip().split() or ["??"])[0])
    print("one-time: raw preload ~4 min + base features ~10 min + noise bank ~2 min, then ~2 s/epoch")


In [ ]:
# v1e progress (re-run anytime)
import subprocess, re

r = subprocess.run(
    ["ssh", "-o", "ConnectTimeout=15", "-o", "BatchMode=yes",
     "jan.rhey.lagana@n003.ai.internal",
     "tail -60 /mnt/jfs_hpc/home/jan.rhey.lagana/vcm/train_v1e.log"],
    capture_output=True, text=True, timeout=60)
# the ssh login banner is ~30 '#' lines — strip it before parsing
out = "\n".join(l for l in r.stdout.splitlines()
                 if l.strip() and not l.strip().startswith("#"))
lines = [l for l in out.splitlines() if re.match(r"ep\s+\d+", l)]
for l in lines[-8:]:
    print(l)
if not lines:
    last = [l for l in out.splitlines() if l.strip()][-2:]
    print("no epochs yet —", " | ".join(last) if last else "log empty")
if "saved ->" in out:
    print("\nDONE — final line:")
    for l in out.splitlines():
        if "saved" in l:
            print(l)


### 10.3 · v1e result (2026-09-28, n003 A100)

| run | where | change | final eval_acc | peak |
|---|---|---|---|---|
| v1c | laptop CPU | + center crop/pad | 0.363 | 0.403 |
| v1d | n003 A100 | + class-weighted CE, warm-start v1c | 0.636 | 0.636 |
| **v1e** | **n003 A100** | **+ real MUSAN noise, 30 ep, warm-start v1d** | **0.750** | **0.753** |

- 30 epochs, **1.86–2.14 s/epoch** (epoch 1 = 8.5 s incl. one-time caches),
  ~1 min wall for the whole run after the ~16 min of data caching
  (raw preload 229 s + base features + noise bank 121 s).
- Trajectory: 0.641 (ep 1) → 0.729 (ep 15) → 0.753 (ep 23) → 0.750 (ep 30)
  — still climbing at the end, plateauing gently around 0.75.
- **Reading:** real noise + longer schedule bought **+0.114 over v1d**.
  The model is now comfortably past the 0.80-target conversation's
  "good enough to demo" line on the clean eval — the remaining honest
  levers are more clips per class (600 vs the 1000–2000 PLAN target) and
  the SNR-robustness axis (eval at SNR 20/10/5 dB per BENCHMARK.md), which
  the MUSAN aug was specifically built to move.


## 11 · v1f — dataset expansion 6,900 → 11,400 clips/split (2026-09-28)

PLAN.md target was **~1,000–2,000 clean clips per class**; v1–v1e all trained
on 600–1,000/class (6,900/split, 13,787 total). The model at 0.750 (v1e) was
plateauing on *data*, not compute — 2.6 s/epoch means more data is cheap.

**Change (committed `06e7b94`):** `src/commands.py` phrase counts 66 → 114
(play_music 6→10, ask_question 7→13, control_lights 8→12, dim_lights 6→10,
set_timer 6→10, set_alarm 6→10, set_temperature 6→10, media_control 10→15,
set_reminder 6→10, make_call 8→14). At 20 voices × 5 rates that is
**11,400 clips/split, 22,800 total**.

**Why the old files stay valid:** `make_dataset.py` numbers clips per class in
(voice, phrase, rate) order and is **resume-safe** (manifest-based skip). The
new phrases were appended at the *end* of each class's phrase list, so clips
1..old-count keep their exact voice/phrase/rate mapping — v1–v1e checkpoints
and the manifest remain consistent. Generation is just `python
src/make_dataset.py` again: it skips the 13,787 manifest rows and synthesizes
only the ~9,000 new combos (measured ~4.4 clips/s, ~35 min).

### 11.1 · Issues hit (measured, not guessed)

| # | Issue (symptom) | Root cause | Fix / workaround |
|---|-----------------|------------|------------------|
| D1 | `Start-Process` launch of the generator hung the tool call to the 120 s timeout (twice) | `Start-Process -PassThru` waits for the child's console handles to close even with `-RedirectStandardOutput` — the generator runs ~35 min | launch and *detach*: the first (hung) call still spawned the process, so the fix was to detect the existing PID via `Win32_Process` cmdline match and let it run, not to re-launch |
| D2 | duplicate phrase in make_call ("call my friend" twice) | hand-typed expansion list | caught by a `len(set(phrases))==len(phrases)` assert in the planning probe; deduped before commit |
| D3 | 13 eval clips had no `.raw` (6 dim_lights, 3 set_alarm, 4 set_timer) | pre-existing from the original decode pass (decode_wav.py is idempotent but those 13 were never decoded) | `decode_wav.py` re-run picks them up automatically (it decodes everything in the manifest missing a `.raw`) — no code change |

**After generation:** re-run `src/decode_wav.py` (mp3 → 16 kHz .raw, idempotent,
~2,900 new files ≈ 2 min), then v1f train on n003 (warm-start v1e, class
weights, real MUSAN noise, 30 epochs — the v1e recipe on the bigger set).


In [ ]:
# v1f dataset generation progress (re-run anytime)
# Local generator: src/make_dataset.py (resume-safe, manifest-based).
# ~4.4 clips/s measured; ~9,000 new clips ≈ 35 min total.
import json, subprocess
from pathlib import Path

root = Path("data/raw")
rows = [json.loads(l) for l in root.joinpath("manifest.jsonl").open(encoding="utf-8") if l.strip()]
print(f"manifest rows: {len(rows)} / 22,800 target")

from collections import Counter
c = Counter((r["split"], r["class"]) for r in rows)
for k in sorted(c):
    print(f"  {k[0]:5s} {k[1]:20s} {c[k]}")

p = subprocess.run(["powershell", "-NoProfile", "-Command",
    "Get-CimInstance Win32_Process -Filter \"Name='python.exe'\" | "
    "Where-Object { $_.CommandLine -like '*make_dataset*' } | Select-Object ProcessId"],
    capture_output=True, text=True)
alive = [l.strip() for l in p.stdout.splitlines() if l.strip().isdigit()]
print("generator alive (PID):", alive or "NOT RUNNING — generation done or needs (re)launch")
if not alive:
    # decode the new mp3s (idempotent)
    print("launching decode_wav.py (mp3 -> .raw, idempotent)...")
    d = subprocess.run([".venv/Scripts/python.exe", "src/decode_wav.py"],
                       capture_output=True, text=True)
    print(d.stdout[-2000:])
    print(d.stderr[-500:])
